In [10]:
import pandas as pd
import matplotlib.pyplot as plt

deliveries = pd.read_csv("C:/Users/donga/Documents/Amazon_Delivery_Analysis/Raw_Data/deliveries.csv")
routes = pd.read_csv("C:/Users/donga/Documents/Amazon_Delivery_Analysis/Raw_Data/routes.csv")

deliveries["promised_days"] = pd.to_numeric(deliveries["promised_days"])
deliveries["actual_days"] = pd.to_numeric(deliveries["actual_days"])

print(f"Total Row: {len(deliveries)}")
deliveries_clean = deliveries.drop_duplicates(keep="first").reset_index(drop=True)
print(f"Clean rows: {len(deliveries_clean)}")

Total Row: 13
Clean rows: 12


In [20]:
merged = deliveries_clean.merge(routes, on="route_id", how="left")
merged["delay_days"] = (merged["actual_days"] - merged["promised_days"])

service_summary = (merged.groupby("service_type" ).agg
    (
        total_delay_days=("delay_days", "sum"),
        delay_incidence_rate=( "delay_days", lambda s: round(100 * (s > 0).sum() / len(s), 2),),
    )
)
print("\nService-type summary: \n")
print(service_summary)


Service-type summary: 

              total_delay_days  delay_incidence_rate
service_type                                        
Express                     12                 66.67
Standard                    22                 83.33


In [28]:
top_route = route_totals.index[0]
top_route_delay = route_totals.iloc[0]
overall_total_delay = merged["delay_days"].sum()

print(f"\nOverall total delay_days: {overall_total_delay}")
print(f"Route with greatest summed delay_days: {top_route} ({top_route_delay} days)")


Overall total delay_days: 34
Route with greatest summed delay_days: R4 (14 days)


In [32]:
monthly_totals = merged.groupby("month", observed=True)["delay_days"].sum()

plt.figure(figsize=(7, 4.5))
plt.bar(monthly_totals.index.astype(str), monthly_totals.values, color="blue")
plt.title("Total Delay Days by Month")
plt.xlabel("Month")
plt.ylabel("Total Delay Days")
plt.tight_layout()
plt.savefig("C:/Users/donga/Documents/Amazon_Delivery_Analysis/Output/python_chart.png", dpi=150)
plt.close()
print("\nSaved chart to python_chart.png")

merged_out = merged.copy()
merged_out["month"] = merged_out["month"].astype(str)
merged_out.to_csv("C:/Users/donga/Documents/Amazon_Delivery_Analysis/Output/clean_data.csv", index=False)
service_summary.to_csv("C:/Users/donga/Documents/Amazon_Delivery_Analysis/Output/python_summary.csv", index=False)
print("Saved clean_data.csv and python_summary.csv")


Saved chart to python_chart.png
Saved clean_data.csv and python_summary.csv
